#### Goal
- To Achive Data Model result by considering All the source systems where W&R Clients are available and GCDS source

#### author
- Aayushi.jain@rabobank.nl

##### flow of logic
- Take W&R clients data from different source systems
- Take GCDS data
- Join on GCDSId to further derive required output

##### Version & Changes

|     Developer |Date	   | PBI/Bug No |	Changes done |
|----------|----------|----------|----------
| Aayushi Jain | 19-JAN-2026 | 14630418 | [Radar Data Model] - Create  new object - Party_Role
| Abhishek Jaiswal | 08-MAY-2026 | 16070041 | Update Radar Data Model Party_Role object from RANZ V4

#####Read Files from GDP

In [0]:
import os
# from pyspark.sql.functions import regexp_extract
from pyspark.sql import SparkSession
app_reg_app_id = os.environ['APP_REG_APP_ID']
ReadStorage = os.environ['GDP_STORAGE_NAME']
TenantId = os.environ['TENANT_ID']
GIC_ReadStorage = os.environ['GDP_SA_STORAGE_NAME']
environment=os.environ['ENV']
radar_datamodel_version_number=3
service_credential = dbutils.secrets.get(scope="connectedsecrets", key = f"appreg-{app_reg_app_id}")
NLS_ReadStorage = os.environ['GDP_NA_STORAGE_NAME']
SARADAR = "saradar" + environment
RANZ_ReadStorage = os.environ['AU_GDP_Defined_Storage_Account']

In [0]:
from RadarUtils import *

In [0]:
dbutils.widgets.text("Load_Date", "")
dbutils.widgets.text("RunType", "daily")  # default is daily

Load_Date = dbutils.widgets.get("Load_Date")
RunType = dbutils.widgets.get("RunType").lower()
if Load_Date:
    RunType="historical"
    
if not Load_Date:
    Load_Date = datetime.today().strftime('%Y%m%d')

print(f"Running for Load Date: {Load_Date}")

In [0]:
party_dataobject='Party_Role'

In [0]:
authenticate_storage_account(ReadStorage)
authenticate_storage_account(GIC_ReadStorage)
authenticate_storage_account(NLS_ReadStorage)
authenticate_storage_account(SARADAR)
authenticate_storage_account(RANZ_ReadStorage)

In [0]:
#Derive the date for which data has to be processes
# import pandas as pd
from datetime import datetime, timedelta
# from pyspark.dbutils import DBUtils
load_dts = f"EDL_LOAD_DTS={Load_Date}*"
print (load_dts)

In [0]:
# print list of strings for loading spark dfs from GDP
gcds_object_list = [
'client_KeyStoreKey',
'client_PartyRole',
"client_Client"
]

# Create TempView for each loading table
for dataobject in gcds_object_list:
    Read_GDP_Defined_DataObjects(Source='GCDS', Dataobject=dataobject,Load_Date=Load_Date)

In [0]:
# print list of strings for loading spark dfs from GDP
gic_object_list = [
'pessoa'
,'vwgic_rdl_pessoa_tipo_cadastro'
]
#gic_load_dts
# Create TempView for each loading table
for dataobject in gic_object_list:
    Read_GDP_Defined_DataObjects(Source='GIC', Dataobject=dataobject,Load_Date=Load_Date)

In [0]:
df_Party_SystemIdentifier = spark.read.parquet(
    f"abfss://radardatamodel@{SARADAR}.dfs.core.windows.net/Party_SystemIdentifier/3/data/{load_dts}/*.parquet"
)
df_Party = spark.read.parquet(
    f"abfss://radardatamodel@{SARADAR}.dfs.core.windows.net/Party/3/data/{load_dts}/*.parquet"
)

In [0]:
df_Party_filter = df_Party.filter((df_Party.PartyIdentifier.like('GCDS%')) | (df_Party.PartyIdentifier.like('GIC%'))).select("PartyIdentifier")

In [0]:
load_df = [
'c_b_party_xref'
,'c_b_contr_rol_party_xref'
,'c_b_contract_xref'
]

#for Dataobject in load_df:
load_ranz_v4_rdm_tables(load_df, Load_Date)#.createOrReplaceTempView(Dataobject)

In [0]:
Ranz_Pep_Evaluation_Static_Dict ={'01'	:'PEP',
'02':	'Close associate of a PEP',
'03':	'Immediate family member of a PEP',
'04':	'Not a PEP'}

sdf_Ranz_Pep_Evaluation_Static_Mapping =spark.createDataFrame([(code, description) for code, description in Ranz_Pep_Evaluation_Static_Dict.items()],['PEP_EVAL_CD','PEP_EVAL_DESCRIPTION'])
sdf_Ranz_Pep_Evaluation_Static_Mapping.createOrReplaceTempView('Ranz_Pep_Evaluation_Static_Mapping')

Ranz_Client_Lifecycle_Status_Dict = {
    "-2": "UNKNOWN",
    "AC": "Active Client",
    "ACPR": "Active-Pending Risk Client",
    "AP": "Applicant Client",
    "BL": "Blocked Client",
    "BL01": "Blocked - Post No Debits",
    "BL02": "Blocked - Post No Credits",
    "BL03": "Blocked - Post No Entries",
    "BL04": "Blocked - Pending Documentation",
    "BL05": "Blocked - Deceased Estate",
    "BL06": "Blocked - Closure Quoted",
    "BL07": "Blocked - Account on Referral List",
    "BL08": "Blocked - Account on Referral List - CR",
    "BL09": "Blocked - Account on Referral List - DR",
    "BL10": "Blocked - Post Credits to Savings a/c",
    "BL11": "Blocked - Post Debits to Current a/c",
    "BL12": "Blocked - Customer Deceased",
    "BL20": "Blocked - Hold Debits: Funds Held as Security",
    "BL50": "Blocked - Setting up of company",
    "BL51": "Blocked - Overdraw not allowed",
    "BL52": "Blocked - Management authorisation",
    "BL53": "Blocked - General Debit Block",
    "BL54": "Blocked - General DR & CR block",
    "BL55": "Blocked - Several blocking codes",
    "BL56": "Blocked - Judicial instructions",
    "BL57": "Blocked - Missing documents",
    "BL58": "Blocked - Temporary blocking",
    "BL59": "Blocked - Multiple blocking",
    "BL60": "Blocked - ATO/IRD request",
    "BL61": "Blocked - Litigation",
    "BL62": "Blocked - Centrelink",
    "BL63": "Blocked - Pending client onboarding",
    "BL64": "Blocked - Incomplete CDD",
    "BL65": "Blocked - Client request block",
    "BL66": "Blocked - Multiple blocking",
    "BL67": "Blocked - Inactive customer",
    "BL68": "Blocked - Overdue remediation",
    "BL69": "Blocked - Bulk account closure remove",
    "BL70": "Blocked - Account being closed",
    "BL71": "Blocked - Pending Client exit",
    "BL72": "Blocked - Pending Client exit DB",
    "BL73": "Blocked - Pending Client exit CR",
    "BL74": "Blocked - Fraud",
    "BL75": "Blocked - For failed authentication",
    "BL76": "Blocked - For disabled end user",
    "BL77": "Blocked - Login Cancelled",
    "BL78": "Blocked - Login Not Used",
    "BL79": "Blocked - Login Cancelled - 1 to 7 years",
    "BL80": "Blocked - Login Cancelled - 0 to 1 years",
    "BL90": "Blocked - Automatic Closure",
    "BL91": "Blocked - Temp Auto Closing",
    "BL99": "Blocked - Account Closure",
    "CL": "Closed Client",
    "DC": "Declined Client",
    "DL": "Deleted",
    "PR": "Prospect Client",
    "WD": "Withdrawn Client",
    "WD01": "Withdrawn - Lost to competitor",
    "WD02": "Withdrawn - Structural conditions could not be met",
    "WD03": "Withdrawn - Not proceeding with funding request",
    "WD04": "Withdrawn - Time constraints",
    "WD05": "Withdrawn - To resubmit",
    "WD06": "Withdrawn - Other",
    "WD07": "Withdrawn - To apply for low cost/no fee account",
    "DC01": "Declined - Credit declined",
    "DC02": "Declined - Unacceptable AML",
    "DC03": "Declined - Other"
}

sdf_RANZ_Client_Lifecycle_Status_Mapping = spark.createDataFrame([(code, desc) for code, desc in Ranz_Client_Lifecycle_Status_Dict.items()],["LIFECYCLE_STATUS_CD", "CLIENT_LIFECYCLE_STATUS_DESCRIPTION"])

sdf_RANZ_Client_Lifecycle_Status_Mapping.createOrReplaceTempView("RANZ_Client_Lifecycle_Status_Mapping")

In [0]:
%sql
create or Replace Temporary view Ranz_ClientDetails as
select distinct
CONCAT('RANZ_',cbp.PKEY_SRC_OBJECT) LocalSystemIdentifier
,cbp.SRC_PARTY_ID
,'RANZ' as Application
,crp.ROLE_TYPE_CD as PartyRoleType
,crp.START_DT as PartyRoleLifecycleStartDate
,crp.END_DT as PartyRoleLifecycleChangeDate
,Lifecycle_Status.CLIENT_LIFECYCLE_STATUS_DESCRIPTION as PartyLifeCycleStatus
FROM c_b_party as cbp
INNER JOIN c_b_contr_rol_party as crp
    ON crp.FK_PARTY_ID = cbp.ROWID_XREF
lEFT JOIN c_b_contract Ranz_Contract on Ranz_Contract.CONTR_ID=crp.FK_CONTR_ID
lEFT JOIN RANZ_Client_Lifecycle_Status_Mapping Lifecycle_Status ON Lifecycle_Status.LIFECYCLE_STATUS_CD=Ranz_Contract.LIFECYCLE_STATUS_CD

In [0]:
%sql
Create or replace temporary view GIC_ClientDetails As
select distinct
p.COD_INSTITUCIONAL
,CONCAT('GIC_', p.COD_INSTITUCIONAL) as LocalSystemIdentifier
,c.DES_TIPO_CADASTRO as PartyRoleType
,c.DTA_CADASTRO AS PartyRoleLifecycleStartDate
,c.DTA_EFETIVACAO AS PartyRoleLifecycleChangeDate
,c.DES_STATUS_TIPO_CADASTRO as PartyLifeCycleStatus
from pessoa p
LEFT JOIN vwgic_rdl_pessoa_tipo_cadastro c on p.COD_INSTITUCIONAL = c.COD_INSTITUCIONAL

In [0]:
%sql
Create or replace temporary view GCDS_Clients As
select distinct
client_keystore.gcid
, client_keystore.KeyStore_value as identifier
, client_keystore.KeyStore_type
, client_partyrole.Party_role AS PartyRoleType 
, client_partyrole.`PartyRole-StartDate` AS PartyRoleLifecycleStartDate
, client_partyrole.`PartyRole-ChangeDate` AS PartyRoleLifecycleChangeDate
, client_partyrole.Life_cycle_status as PartyLifeCycleStatus 
from client_KeyStoreKey client_keystore 
inner join client_Client client on client.GCID = client_keystore.GCID
left join client_PartyRole client_partyrole on client_keystore.GCID = client_partyrole.GCID

In [0]:
%sql
Create or replace temporary view party As
select distinct
CASE 
     WHEN gcds_client.identifier = gic_clientdetails.COD_INSTITUCIONAL AND gcds_client.KeyStore_type = 'GIC' THEN gic_clientdetails.LocalSystemIdentifier
     WHEN gcds_client.identifier = ranz.SRC_PARTY_ID AND gcds_client.KeyStore_type = 'CB RANZ' THEN ranz.LocalSystemIdentifier
     else CONCAT('GCDS_',gcds_client.gcid)
END AS LocalSystemIdentifier
,COALESCE(gcds_client.PartyRoleType,gic_clientdetails.PartyRoleType,ranz.PartyRoleType) as PartyRoleType
,COALESCE(gcds_client.PartyRoleLifecycleStartDate,gic_clientdetails.PartyRoleLifecycleStartDate,ranz.PartyRoleLifecycleStartDate) as PartyRoleLifecycleStartDate
,COALESCE(gcds_client.PartyRoleLifecycleChangeDate,gic_clientdetails.PartyRoleLifecycleChangeDate,ranz.PartyRoleLifecycleChangeDate) as PartyRoleLifecycleChangeDate
,COALESCE(gcds_client.PartyLifeCycleStatus,gic_clientdetails.PartyLifeCycleStatus,ranz.PartyLifeCycleStatus) as PartyLifeCycleStatus
from GCDS_Clients gcds_client
left join GIC_ClientDetails gic_clientdetails
on gcds_client.identifier = gic_clientdetails.COD_INSTITUCIONAL
and gcds_client.KeyStore_type = 'GIC'
left join Ranz_ClientDetails ranz on gcds_client.identifier = ranz.SRC_PARTY_ID 
     and gcds_client.KeyStore_type = 'CB RANZ'   

In [0]:
%sql
Create or replace temporary view NonGCDS_UniqueParty As
select distinct gic_clientdetails.LocalSystemIdentifier,  gic_clientdetails.PartyRoleType
     ,gic_clientdetails.PartyRoleLifecycleStartDate
     ,gic_clientdetails.PartyRoleLifecycleChangeDate
     ,gic_clientdetails.PartyLifeCycleStatus
from GIC_ClientDetails gic_clientdetails
left anti join party gcds_party
on gic_clientdetails.LocalSystemIdentifier = gcds_party.LocalSystemIdentifier

union

select distinct 
ranz.LocalSystemIdentifier
,ranz.PartyRoleType
,ranz.PartyRoleLifecycleStartDate
,ranz.PartyRoleLifecycleChangeDate
,ranz.PartyLifeCycleStatus
From Ranz_ClientDetails ranz
left anti join party gcds_party
on ranz.LocalSystemIdentifier = gcds_party.LocalSystemIdentifier

In [0]:
df_party_role_gic_gcds= spark.sql("""select *
from party
union
select *
from NonGCDS_UniqueParty""")


In [0]:
df_party_role_ = add_party_identifier(df_party_role_gic_gcds, df_Party_SystemIdentifier)


In [0]:
df_party_role = df_Party_filter.join(df_party_role_, "PartyIdentifier", how="left").distinct()

In [0]:
if RunType == "historical":
    save_to_saradar_storage_account(df_party_role, party_dataobject, radar_datamodel_version_number, environment,Load_Date)
else:
    save_to_saradar_storage_account(df_party_role, party_dataobject, radar_datamodel_version_number, environment)